# floret vectors on a Colab T4

Trains floret-mode fastText vectors (subword n-grams hashed into a compact Bloom table) on GPU with PyTorch, and benchmarks against CPU `floret` on both **speed** and **quality**.

**Runtime -> Change runtime type -> T4 GPU** before running.


## What changed since the previous bundle

Two fixes, both load-bearing. Read them or the numbers below will not make sense.

### 1. Input-gradient normalization (this is why the last run collapsed)

The previous bundle divided the input-row gradient by the bag size $L$. fastText does **not** do this for cbow/skipgram — `Model::update` normalizes only in supervised mode. It matters: a cbow window averages ~240 subword rows, so the $1/L$ gradient was ~240x too small to bootstrap while `wo` starts at zero. `hidden` never escaped ~$10^{-4}$.

Symptom you saw: every cosine 1.00, `king` -> `peaking`, `cheesemaking`. Measured on a 25MB text8 slice, removing the $1/L$:

| | before | after |
|---|---|---|
| loss | 4.018 | **2.562** |
| WS353 | -0.123 | **0.193** |
| mean pairwise cos | 1.000 | **0.584** |

### 2. Batch size is now a correctness knob

With the un-normalized update, a large batch accumulates hundreds of full-size gradients onto a single row before applying them, against a stale `hidden`. At `--batch 65536` this diverges outright (I measured loss -> 1e14). **Your last run used `--batch 262144`.**

Use **4096-16384**; the default is now 8192. If loss explodes, halve it.


## Why this is fast

Two things, neither of which is a custom CUDA kernel (none is used — everything is stock PyTorch ops):

1. **Per-batch word dedup.** Every occurrence of a word in a batch has the same input vector, so its subword bag is gathered once and indexed. Exact, not an approximation. On text8, 65536 context tokens collapse to ~3900 unique words.
2. **Hand-written gradients.** Autograd cost 139ms/batch on the output side plus a 33ms dense optimizer sweep, because it allocates and zeroes dense `(rows, dim)` gradient buffers every step and then updates all 121k rows. The SGNS gradient is analytic and touches few rows, so it is computed directly and scattered with `index_add_`:

$$\alpha = \eta\,(\text{label}-\sigma(w_o\cdot h)),\quad w_o \mathrel{+}= \alpha h,\quad g \mathrel{+}= \alpha w_o^{\text{old}},\quad \forall r \in \text{bag}: W_r \mathrel{+}= g$$


In [ ]:
# CELL 1 -- confirm the hardware
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, os
print('torch', torch.__version__, '| cuda', torch.version.cuda,
      '| capability', torch.cuda.get_device_capability(0),
      '| vCPUs', os.cpu_count())

In [ ]:
# CELL 2 -- dependencies (Colab already has torch + numpy)
!pip -q install mmh3 floret

In [ ]:
# CELL 3 -- upload floret_torch_bundle.zip (build locally: bash colab/make_bundle.sh)
from google.colab import files
files.upload()
!unzip -o -q floret_torch_bundle.zip && ls floret_torch bench

### Parity gate
This trains a tiny CPU floret model and checks our MurmurHash seed, the 128-bit -> 4x uint32 key split, n-gram extraction and the `.floret` format against it. If this fails, nothing downstream is meaningful.

In [ ]:
# CELL 4 -- hashing parity vs CPU floret
!python -m pytest tests/test_parity.py -q

In [ ]:
# CELL 5 -- corpus + evaluation set
!mkdir -p data out data/wordsim353
!test -f data/text8 || (curl -sL -o data/text8.zip http://mattmahoney.net/dc/text8.zip && unzip -o -q -d data data/text8.zip)
!curl -sfL -o data/wordsim353/raw.txt https://raw.githubusercontent.com/mfaruqui/eval-word-vectors/master/data/word-sim/EN-WS-353-ALL.txt
!ls -la data/text8 && wc -l < data/wordsim353/raw.txt

## Training
Cell 6 is the reference GPU run. Cell 7 is identical except the tables are stored fp16, and it reuses cell 6's binarization cache so the `train=` timings compare like for like.

In [ ]:
# CELL 6 -- GPU cbow, fp32
!python -m floret_torch.train \
  --input data/text8 --output out/gpu_cbow_fp32 \
  --model cbow --dim 300 --minn 4 --maxn 5 --hashCount 2 --bucket 50000 \
  --epoch 1 --lr 0.05 --ws 5 --neg 5 --minCount 5 --t 1e-4 \
  --batch 8192 --slab 4000000 --dtype fp32 --device cuda --seed 0

### fp16
Storage of both tables is fp16; **all** logit and gradient arithmetic stays fp32 on the gathered slices. That split is deliberate: an SGNS update is ~$10^{-5}$ against weights ~$3\times10^{-3}$, where fp16's ulp is ~$4\times10^{-6}$. Storing fp16 is fine; *accumulating* in fp16 would quietly stop learning once `lr` decays.

Your own T4 measurements set the expectation — this workload is gather-bound, not matmul-bound:

| op | fp32 | fp16 |
|---|---|---|
| matmul 2048^3 | 5.02 ms | 0.55 ms (9x) |
| embedding_bag fwd+bwd | 28.27 ms | 24.29 ms (**1.16x**) |

So expect roughly 10-15% end to end, not 9x. Cell 7 gets the real number.

In [ ]:
# CELL 7 -- GPU cbow, fp16 (same seed/params, reuses cell 6's cache)
!python -m floret_torch.train \
  --input data/text8 --output out/gpu_cbow_fp16 \
  --model cbow --dim 300 --minn 4 --maxn 5 --hashCount 2 --bucket 50000 \
  --epoch 1 --lr 0.05 --ws 5 --neg 5 --minCount 5 --t 1e-4 \
  --batch 8192 --slab 4000000 --dtype fp16 --device cuda --seed 0 \
  --workdir out/gpu_cbow_fp32.work --reuse-cache

### CPU baseline
**Caveat that must travel with any speedup number:** a Colab VM has ~2 vCPUs. The same run on my 4-core laptop took **188.1s**; your 2-thread Colab run took **371.3s**. Quote the ratio against a real core count, not against Colab's.

In [ ]:
# CELL 8 -- CPU baseline, identical hyperparameters
import os
NCPU = os.cpu_count()
!python bench/cpu_baseline.py --input data/text8 --output out/cpu_cbow \
  --model cbow --dim 300 --epoch 1 --thread {NCPU}

## Quality gate
This decides whether the vectors are usable. Gate: **GPU WS353 >= CPU WS353 - 0.05**.

Reference points I measured locally on full text8: CPU **0.2875**, old broken GPU run **0.1588**.

Eyeball check: `three` should pull in four/five/two/six and `music` should pull in musical/musicians. If neighbours are all suffix twins and mean cosine is ~1.0, it is undertrained — go to cell 11.

In [ ]:
# CELL 9 -- quality: GPU (fp32 and fp16) vs CPU
!python bench/eval.py --vec out/gpu_cbow_fp32.vec --compare out/cpu_cbow.vec --topn 8
print('=' * 70)
!python bench/eval.py --vec out/gpu_cbow_fp16.vec --compare out/cpu_cbow.vec --topn 8

In [ ]:
# CELL 10 -- optional: skipgram. One update per PAIR rather than per center,
# so ~6x more updates and ~6x larger negative tensors -> smaller batch.
!python -m floret_torch.train \
  --input data/text8 --output out/gpu_sg \
  --model skipgram --dim 300 --bucket 50000 --epoch 1 --lr 0.05 \
  --batch 4096 --slab 4000000 --dtype fp32 --device cuda --seed 0 \
  --workdir out/gpu_cbow_fp32.work --reuse-cache
!python bench/eval.py --vec out/gpu_sg.vec --compare out/cpu_cbow.vec --topn 8

### If quality is short of the gate
Spend the speed surplus on **epochs** first — it is the cheapest lever and the one with headroom. One epoch took ~10s on your T4 against a 371s CPU run, so 5 epochs still leaves a ~35x margin. Only if that stalls should you touch `--lr`.

In [ ]:
# CELL 11 -- more epochs
!python -m floret_torch.train \
  --input data/text8 --output out/gpu_cbow_e5 \
  --model cbow --dim 300 --bucket 50000 --epoch 5 --lr 0.05 \
  --batch 8192 --slab 4000000 --dtype fp32 --device cuda --seed 0 \
  --workdir out/gpu_cbow_fp32.work --reuse-cache
!python bench/eval.py --vec out/gpu_cbow_e5.vec --compare out/cpu_cbow.vec --topn 8

In [ ]:
# CELL 12 -- optional: batch x dtype tradeoff in one table.
# Trains once per combination, so keep the lists short.
!python bench/sweep.py --input data/text8 --workdir out/gpu_cbow_fp32.work \
  --batches 4096,8192,16384 --dtypes fp32,fp16 --epoch 1

In [ ]:
# CELL 13 -- download. The .floret table loads into spaCy v3.2+ with:
#   spacy init vectors en vectors.floret out_dir --mode floret
from google.colab import files
files.download('out/gpu_cbow_fp32.floret')
files.download('out/gpu_cbow_fp32.vec')

## What to send back
- the `TIMING` line from cells 6, 7, 10
- the `CPU_TRAIN_SECONDS` line from cell 8
- the full output of cell 9 (both WS353 numbers and the neighbour lists)

That settles both open questions: whether fp16 earns its place, and whether the GPU vectors clear the CPU quality bar.